# 📊 Planilha Embriologia Validation: Local DuckDB vs AWS Athena (Prod)

This notebook automates the validation and comparison of the three versions of `planilha_embriologia` (combined, fresh, and fet) between the local DuckDB database (`huntington_data_lake.duckdb`) and production AWS Athena (`silver_embriologia_prod`).

### Rules:
- **Rule**: Each code cell performing queries must explicitly open and close database connections.


In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Database Configurations
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_embriologia_prod'

print("Configuration set.")
print(f"Local DuckDB Path: {os.path.abspath(DUCKDB_PATH)}")
print(f"AWS Athena Schema: {ATHENA_DB}")


Configuration set.
Local DuckDB Path: G:\My Drive\projetos_individuais\Huntington\database\huntington_data_lake.duckdb
AWS Athena Schema: silver_embriologia_prod


## 🔌 Connection Test
Testing connection to both DuckDB and Athena, opening and closing the connection immediately.


In [2]:
try:
    with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
        duck_ok = conn.execute("SELECT 1 as test").fetchone()[0] == 1
    print("✅ Local DuckDB Connection: OK")
except Exception as e:
    print(f"❌ Local DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            athena_ok = cur.fetchone()[0] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False


✅ Local DuckDB Connection: OK


✅ AWS Athena Connection: OK


## 🟢 Part 1: Reconcile fresh Table (Local planilha_embriologia_fresh vs Athena fresh)

In this section, we compare schemas, counts, distinct keys, and outcome sums.


In [3]:
# 1. Compare fresh schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_fresh LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM fresh LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Fresh columns in Local only: {only_local}")
print(f"Fresh columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


Fresh columns in Local only: ['data_da_puncao', 'file_name', 'idade_espermatozoide', 'no_biopsiados', 'nome_da_paciente', 'origem_espermatozoide', 'qtd_blasto_tq_a_e_b', 'sheet_name', 'tipo_1', 'tipo_de_inseminacao', 'tipo_espermatozoide']
Fresh columns in Athena only: ['bronze_ingested_at', 'data_puncao', 'idade_mulher', 'n_biopsiados', 'obs', 'qtd_blasto_tq', 'result', 'source_file', 'source_format', 'source_sheet', 'tipo_inseminacao', 'tipo_tratamento', 'unidade', 'year']
Common columns: 17


In [4]:
# 2. Compare Counts and Keys for fresh
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_fresh
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM fresh
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique PIN', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_pin'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_pin'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
display(df_counts)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Total Rows,13467,13523,-56,99.584169
1,Unique PIN,9586,9931,-345,96.401001
2,Unique Prontuario,9055,9079,-24,99.734953


In [5]:
# 3. Compare Outcome Values for fresh
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_outcomes = conn.execute('''
        SELECT 
            SUM(qtd_blasto) as sum_qtd_blasto,
            SUM(qtd_blasto_tq_a_e_b) as sum_qtd_blasto_tq,
            SUM(no_biopsiados) as sum_no_biopsiados,
            SUM(qtd_analisados) as sum_qtd_analisados,
            SUM(qtd_normais) as sum_qtd_normais
        FROM silver.planilha_embriologia_fresh
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_outcomes = pd.read_sql('''
        SELECT 
            SUM(qtd_blasto) as sum_qtd_blasto,
            SUM(qtd_blasto_tq) as sum_qtd_blasto_tq,
            SUM(n_biopsiados) as sum_no_biopsiados,
            SUM(qtd_analisados) as sum_qtd_analisados,
            SUM(qtd_normais) as sum_qtd_normais
        FROM fresh
    ''', conn)

df_outcomes = pd.DataFrame({
    'Outcome Metric': ['Sum qtd_blasto', 'Sum qtd_blasto_tq', 'Sum no_biopsiados', 'Sum qtd_analisados', 'Sum qtd_normais'],
    'Local (DuckDB)': [
        local_outcomes.loc[0, 'sum_qtd_blasto'], local_outcomes.loc[0, 'sum_qtd_blasto_tq'],
        local_outcomes.loc[0, 'sum_no_biopsiados'], local_outcomes.loc[0, 'sum_qtd_analisados'],
        local_outcomes.loc[0, 'sum_qtd_normais']
    ],
    'Athena (Prod)': [
        prod_outcomes.loc[0, 'sum_qtd_blasto'], prod_outcomes.loc[0, 'sum_qtd_blasto_tq'],
        prod_outcomes.loc[0, 'sum_no_biopsiados'], prod_outcomes.loc[0, 'sum_qtd_analisados'],
        prod_outcomes.loc[0, 'sum_qtd_normais']
    ]
})
df_outcomes['Delta'] = df_outcomes['Local (DuckDB)'] - df_outcomes['Athena (Prod)']
df_outcomes['Match Rate %'] = (1 - (df_outcomes['Delta'].abs() / df_outcomes['Local (DuckDB)'])) * 100
display(df_outcomes)


,Outcome Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Sum qtd_blasto,39596.0,39763,-167.0,99.578240
1,Sum qtd_blasto_tq,29672.0,29797,-125.0,99.578727
2,Sum no_biopsiados,14526.0,24292,-9766.0,32.768828
3,Sum qtd_analisados,21471.0,21606,-135.0,99.371245
4,Sum qtd_normais,7472.0,7541,-69.0,99.076552


## 🔵 Part 2: Reconcile fet Table (Local planilha_embriologia_fet vs Athena fet)

In this section, we compare schemas, counts, distinct keys, and outcome value distributions.


In [6]:
# 1. Compare fet schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_fet LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM fet LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Fet columns in Local only: {only_local}")
print(f"Fet columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


Fet columns in Local only: ['data_da_fet', 'data_de_nasc', 'dia_et', 'file_name', 'idade_do_cong_de_embriao', 'no_da_transfer_1a_2a_3a', 'no_et', 'no_nascidos', 'nome_da_paciente', 'preparo_para_transferencia', 'sheet_name', 'tipo_1', 'tipo_da_doacao', 'tipo_de_fet', 'tipo_de_tratamento', 'tipo_do_resultado']
Fet columns in Athena only: ['bronze_ingested_at', 'data_et', 'data_fet', 'idade_cong_embriao', 'n_da_transfer', 'n_et', 'n_nascidos', 'preparo_transferencia', 'source_file', 'source_format', 'source_sheet', 'tipo_doacao', 'tipo_fet', 'tipo_resultado', 'tipo_tratamento', 'unidade', 'year']
Common columns: 10


In [7]:
# 2. Compare Counts and Keys for fet
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_fet
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM fet
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique PIN', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_pin'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_pin'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
display(df_counts)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Total Rows,23240,11850,11390,50.989673
1,Unique PIN,13196,8954,4242,67.853895
2,Unique Prontuario,11775,7858,3917,66.734607


In [8]:
# 3. Compare Outcome Values for fet (distributions of result, gravidez_clinica, gravidez_bioquimica, no_nascidos)
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_res = conn.execute("SELECT result, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY result").df()
    local_gc = conn.execute("SELECT gravidez_clinica, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY gravidez_clinica").df()
    local_gb = conn.execute("SELECT gravidez_bioquimica, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY gravidez_bioquimica").df()
    local_nn = conn.execute("SELECT no_nascidos, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY no_nascidos").df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_res = pd.read_sql("SELECT result, COUNT(*) as cnt FROM fet GROUP BY result", conn)
    prod_gc = pd.read_sql("SELECT gravidez_clinica, COUNT(*) as cnt FROM fet GROUP BY gravidez_clinica", conn)
    prod_gb = pd.read_sql("SELECT gravidez_bioquimica, COUNT(*) as cnt FROM fet GROUP BY gravidez_bioquimica", conn)
    prod_nn = pd.read_sql("SELECT n_nascidos as no_nascidos, COUNT(*) as cnt FROM fet GROUP BY n_nascidos", conn)

def merge_dist(local_df, prod_df, col_name):
    local_df.columns = [col_name, 'Local']
    prod_df.columns = [col_name, 'Athena']
    
    # Clean and cast the join keys to string
    local_df[col_name] = local_df[col_name].astype(str).str.strip().str.replace('.0', '', regex=False).str.upper()
    prod_df[col_name] = prod_df[col_name].astype(str).str.strip().str.replace('.0', '', regex=False).str.upper()
    
    # Re-group in case casting caused duplicate categories
    local_df = local_df.groupby(col_name, as_index=False).sum()
    prod_df = prod_df.groupby(col_name, as_index=False).sum()
    
    merged = pd.merge(local_df, prod_df, on=col_name, how='outer').fillna(0)
    merged['Delta'] = merged['Local'] - merged['Athena']
    return merged

print("--- Result Distribution ---")
display(merge_dist(local_res, prod_res, 'result'))

print("\n--- Gravidez Clinica Distribution ---")
display(merge_dist(local_gc, prod_gc, 'gravidez_clinica'))

print("\n--- Gravidez Bioquimica Distribution ---")
display(merge_dist(local_gb, prod_gb, 'gravidez_bioquimica'))

print("\n--- No Nascidos Distribution ---")
display(merge_dist(local_nn, prod_nn, 'no_nascidos'))


--- Result Distribution ---


,result,Local,Athena,Delta
0,ABNORMAL FERT,2,0.0,2.0
1,CANCELADO,1,0.0,1.0
2,CANCELLATION,62,67.0,-5.0
3,DONOR,1069,0.0,1069.0
4,EGG FREEZING,2063,0.0,2063.0
5,EMBRYO TRANSFER,5484,5486.0,-2.0
6,EMBRYO VITRI,5891,336.0,5555.0
7,IMMATURE EGGS,6,0.0,6.0
8,NEGATIVO,2827,2396.0,431.0
9,NO EGGS,17,0.0,17.0



--- Gravidez Clinica Distribution ---


,gravidez_clinica,Local,Athena,Delta
0,0,2303,2307,-4
1,1,2581,2582,-1
2,NONE,18204,6809,11395
3,X,152,152,0



--- Gravidez Bioquimica Distribution ---


,gravidez_bioquimica,Local,Athena,Delta
0,,0.0,1,-1.0
1,0,275.0,1907,-1632.0
2,1,490.0,2998,-2508.0
3,NONE,22394.0,6792,15602.0
4,X,81.0,152,-71.0



--- No Nascidos Distribution ---


,no_nascidos,Local,Athena,Delta
0,0,7.0,7.0,0.0
1,1,162.0,162.0,0.0
2,2,19.0,19.0,0.0
3,NAN,0.0,11662.0,-11662.0
4,NONE,23052.0,0.0,23052.0


## 🟣 Part 3: Reconcile planilha_embriologia_combined (Local vs Athena)

In this section, we compare schemas, counts, unique keys, and outcome values in the combined table.


In [9]:
# 1. Compare combined schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_combined LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM planilha_embriologia_combined LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Combined columns in Local only: {only_local}")
print(f"Combined columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


Combined columns in Local only: ['fet_file_name', 'fet_idade_do_cong_de_embriao', 'fet_no_da_transfer_1a_2a_3a', 'fet_preparo_para_transferencia', 'fet_sheet_name', 'fet_tipo_1', 'fet_tipo_da_doacao', 'fet_tipo_de_fet', 'fet_tipo_de_tratamento', 'fresh_data_da_puncao', 'fresh_file_name', 'fresh_idade_espermatozoide', 'fresh_no_biopsiados', 'fresh_qtd_blasto_tq_a_e_b', 'fresh_sheet_name', 'fresh_tipo_1', 'fresh_tipo_de_inseminacao']
Combined columns in Athena only: ['fet_idade_cong_embriao', 'fet_no_da_transfer', 'fet_preparo_transferencia', 'fet_source_file', 'fet_source_sheet', 'fet_tipo_doacao', 'fet_tipo_fet', 'fet_tipo_tratamento', 'fresh_data_puncao', 'fresh_n_biopsiados', 'fresh_qtd_blasto_tq', 'fresh_source_file', 'fresh_source_sheet', 'fresh_tipo_inseminacao', 'fresh_tipo_tratamento']
Common columns: 29


In [10]:
# 2. Compare Counts and Keys for combined
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_combined
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM planilha_embriologia_combined
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
display(df_counts)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Total Rows,27093,20664,6429,76.270623
1,Unique Prontuario,13518,10951,2567,81.010505


In [11]:
# 3. Compare Outcome Values for combined
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_outcomes = conn.execute('''
        SELECT 
            SUM(fresh_qtd_blasto) as sum_fresh_qtd_blasto,
            SUM(fresh_qtd_blasto_tq_a_e_b) as sum_fresh_qtd_blasto_tq,
            SUM(fresh_no_biopsiados) as sum_fresh_no_biopsiados,
            SUM(fresh_qtd_analisados) as sum_fresh_qtd_analisados,
            SUM(fresh_qtd_normais) as sum_fresh_qtd_normais
        FROM silver.planilha_embriologia_combined
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_outcomes = pd.read_sql('''
        SELECT 
            SUM(fresh_qtd_blasto) as sum_fresh_qtd_blasto,
            SUM(fresh_qtd_blasto_tq) as sum_fresh_qtd_blasto_tq,
            SUM(fresh_n_biopsiados) as sum_fresh_no_biopsiados,
            SUM(fresh_qtd_analisados) as sum_fresh_qtd_analisados,
            SUM(fresh_qtd_normais) as sum_fresh_qtd_normais
        FROM planilha_embriologia_combined
    ''', conn)

df_outcomes = pd.DataFrame({
    'Outcome Metric': ['Sum fresh_qtd_blasto', 'Sum fresh_qtd_blasto_tq', 'Sum fresh_no_biopsiados', 'Sum fresh_qtd_analisados', 'Sum fresh_qtd_normais'],
    'Local (DuckDB)': [
        local_outcomes.loc[0, 'sum_fresh_qtd_blasto'], local_outcomes.loc[0, 'sum_fresh_qtd_blasto_tq'],
        local_outcomes.loc[0, 'sum_fresh_no_biopsiados'], local_outcomes.loc[0, 'sum_fresh_qtd_analisados'],
        local_outcomes.loc[0, 'sum_fresh_qtd_normais']
    ],
    'Athena (Prod)': [
        prod_outcomes.loc[0, 'sum_fresh_qtd_blasto'], prod_outcomes.loc[0, 'sum_fresh_qtd_blasto_tq'],
        prod_outcomes.loc[0, 'sum_fresh_no_biopsiados'], prod_outcomes.loc[0, 'sum_fresh_qtd_analisados'],
        prod_outcomes.loc[0, 'sum_fresh_qtd_normais']
    ]
})
df_outcomes['Delta'] = df_outcomes['Local (DuckDB)'] - df_outcomes['Athena (Prod)']
df_outcomes['Match Rate %'] = (1 - (df_outcomes['Delta'].abs() / df_outcomes['Local (DuckDB)'])) * 100
display(df_outcomes)


,Outcome Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Sum fresh_qtd_blasto,39596.0,39763,-167.0,99.578240
1,Sum fresh_qtd_blasto_tq,29672.0,29797,-125.0,99.578727
2,Sum fresh_no_biopsiados,14526.0,24292,-9766.0,32.768828
3,Sum fresh_qtd_analisados,21471.0,21606,-135.0,99.371245
4,Sum fresh_qtd_normais,7472.0,7541,-69.0,99.076552
